In [2]:
! pip install tensorflow opencv-python matplotlib scikit-learn scikit-image

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Verification
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import sklearn
import skimage

import os
import random
import numpy as np

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("Scikit-Learn:", sklearn.__version__)
print("Scikit-Image:", skimage.__version__)

TensorFlow: 2.20.0
OpenCV: 4.13.0
Scikit-Learn: 1.8.0
Scikit-Image: 0.26.0


In [4]:
plt.imshow??

Signature:
plt.imshow(
    X: 'ArrayLike | PIL.Image.Image',
    cmap: 'str | Colormap | None' = None,
    norm: 'str | Normalize | None' = None,
    *,
    aspect: "Literal['equal', 'auto'] | float | None" = None,
    interpolation: 'str | None' = None,
    alpha: 'float | ArrayLike | None' = None,
    vmin: 'float | None' = None,
    vmax: 'float | None' = None,
    colorizer: 'Colorizer | None' = None,
    origin: "Literal['upper', 'lower'] | None" = None,
    extent: 'tuple[float, float, float, float] | None' = None,
    interpolation_stage: "Literal['data', 'rgba', 'auto'] | None" = None,
    filternorm: 'bool' = True,
    filterrad: 'float' = 4.0,
    resample: 'bool | None' = None,
    url: 'str | None' = None,
    data=None,
    **kwargs,
) -> 'AxesImage'
Docstring:
Display data as an image, i.e., on a 2D regular raster.

The input may either be actual RGB(A) data, or 2D scalar data, which
will be rendered as a pseudocolor image. For displaying a grayscale
image, set up the col

In [2]:
from tensorflow.keras.models import Model 
from tensorflow.keras.layers import Layer, Conv2D, MaxPooling2D, Input, Flatten #Dense, Dropout, BatchNormalization
import tensorflow as tf 

In [6]:
# Model (inputs=[inputimage, verification_image], outputs=[1, 0])

In [7]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [8]:
len(gpus)


0

In [9]:
gpus

[]

In [10]:
# I tried to work with the GPU but it seems that it is not being detected. I will continue with the CPU for now. 
# TensorFlow's GPU support on native Windows was effectively dropped after TensorFlow 2.10.

Create Folder Structures
--------

In [3]:
# Setup Paths
POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')

In [15]:
# Create folders
os.makedirs(POS_PATH)
os.makedirs(NEG_PATH)
os.makedirs(ANC_PATH)

FileExistsError: [WinError 183] Impossible de créer un fichier déjà existant: 'data\\positive'

Collect Positives And Anchors 
---------

In [ ]:
# Uncompress the dataset 'Labelled Faces in the Wild'
! tar -xf archive.zip

In [24]:
# Moving LFW Images to the folder : data/negative
for directory in os.listdir(r'lfw-deepfunneled/lfw-deepfunneled'):
    for file in os.listdir(os.path.join(os.path.join(r'lfw-deepfunneled/lfw-deepfunneled', directory))):
        EX_PATH = os.path.join(r'lfw-deepfunneled/lfw-deepfunneled', directory, file)
        NEW_PATH = os.path.join(NEG_PATH, file)
        os.replace(EX_PATH, NEW_PATH)


In [4]:
# Importing uuid library to generate unique names for the images captured from the webcam
import uuid

In [5]:
# Collecte Positive and Anchor Classes

cap = cv2.VideoCapture(0) # Establish a connection to the webcam
while cap.isOpened(): 
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = frame[120:120+250, 200:200+250, : ] # Crop the image to a square shape (250x250) from the center of the frame
    
    # Collect Anchors
    if cv2.waitKey(1) & 0xFF == ord('a'):
        imagename = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1())) # Generate a unique name for the image
        cv2.imwrite(imagename, frame) # Save the image to the anchor folder
    
    # Collect Positives 
    if cv2.waitKey(1) & 0xFF == ord('p'):   
        imagename = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1())) # Generate a unique name for the image
        cv2.imwrite(imagename, frame) # Save the image to the positive folder   
    
    
    
    cv2.imshow('Image Collection', frame) # Show image back to screen
    
    # Breaking gracefully
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release() # Realease the webcam
cv2.destroyAllWindows() # Close(destry) the image show frame

In [ ]:

for i in range(5):
    cap = cv2.VideoCapture(i)
    if cap.isOpened():
        print(f"Camera found at index {i}")
    else:
        print(f"No camera at index {i}")
    cap.release()
    
# To know the camera in wich index so we can choose the right number(in VideoCapture)

Camera found at index 0
No camera at index 1
No camera at index 2
No camera at index 3
No camera at index 4


Load and Preprocess Images
---------

In [6]:
# Get Image Directories
anchor = tf.data.Dataset.list_files(ANC_PATH+'/*.jpg').take(210)
positive = tf.data.Dataset.list_files(POS_PATH+'/*.jpg').take(210)
negative = tf.data.Dataset.list_files(NEG_PATH+'/*.jpg').take(210)

In [7]:
#Preprocessing - Scale and Resize 

def preprocess(file_path):
    # Read in image from file path
    byte_img = tf.io.read_file(file_path)
    # Load in the image 
    img = tf.io.decode_jpeg(byte_img)
    # Preprocessing steps - resizing the image to be 100x100x3
    img = tf.image.resize(img, (100,100))
    # Scale image to be between 0 and 1 
    img = img / 255.0
    
    return img

In [8]:
# Create Labelled Dataset
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor))))) # 1 = positive match
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor))))) # 0 = negative match
dataset = positives.concatenate(negatives)

In [9]:
 # (anchor , positive) => 1,1,1,1,1
 # (anchor , negative) => 0,0,0,0,0

In [10]:
samples = dataset.as_numpy_iterator()
samples.next()

(b'data\\anchor\\27c311c1-6f13-11f1-bab6-c018034eab96.jpg',
 b'data\\positive\\a09ae0f0-6f13-11f1-a46f-c018034eab96.jpg',
 np.float32(1.0))

In [11]:
# Built Train and Test Prtition
def preprocess_twin(input_img, validation_img, label):
    return(preprocess(input_img), preprocess(validation_img), label)

In [21]:
# Build dataloader pipeline
data = dataset.map(preprocess_twin)
data = data.cache()
data = data.shuffle(buffer_size=1024)

In [38]:
# Training partition
train_data = data.take(round(len(data)*.7)) # round(len(data)*.7) = number of samples in the training set, which is 70% of the total dataset. This is a common practice in machine learning to split the dataset into training and testing sets, where the training set is used to train the model and the testing set is used to evaluate its performance.
train_data = train_data.batch(16) # batch size of 16 means that the model will process 16 samples at a time before updating the weights. This is a common practice in deep learning to balance between memory usage and training speed.
train_data = train_data.prefetch(8) # Prefetching allows the data loading to be done in the background while the model is training, which can improve training performance.

In [40]:
# Test partition
test_data = data.skip(round(len(data)*.7)) # skip the first 70% of the dataset to get the remaining 30% for testing
test_data = test_data.take(round(len(data)*.3)) # take the next 30% 
test_data = test_data.batch(16) 
test_data = test_data.prefetch(8) 